# Chapter 21 — Network Change Automation with AI
## From "I want OSPF authentication" to a safe, validated, rollback-ready config

Gartner says **80% of network outages are caused by human error** — and the most
dangerous moment is the change window. 2 AM, maintenance slot, production router,
tired engineer.

Five things go wrong in every manual change process:

| Failure Mode | What Happens | AI Solution |
|---|---|---|
| No pre-validation | Mental review misses interactions | LLM validates against current state |
| No impact analysis | "Just a static route" black-holes 400 users | Digital twin simulates effect first |
| No automated rollback | Manual panic revert causes second error | Checkpoint + auto-rollback |
| No real-time verification | Change applied, but did it work? | Post-change verification sweep |
| Incomplete docs | Filled from memory, audit trail lost | Auto-generated change log |

This notebook shows how AI solves all five — with code you can run today.

> **No GPU needed.** Uses only the Anthropic API (Haiku for validation, Sonnet for config generation).

### Install
```
pip install anthropic
```


## Setup — API Client and Network State

In [ ]:
import json, re, time, copy
from dataclasses import dataclass
from anthropic import Anthropic

# ── API key setup ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=api_key)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system="You are a senior network automation engineer."):
    """Single-turn helper — returns plain text."""
    resp = client.messages.create(
        model=model, max_tokens=1024, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()

# ── Simulated network inventory ───────────────────────────────────────────────
# This is what you'd pull from NETCONF/RESTCONF/Napalm in production
NETWORK_INVENTORY = {
    "R1": {
        "platform": "Cisco IOS-XE",
        "interfaces": {"Gi0/0": {"ip": "10.0.12.1/30", "area": 0},
                       "Gi0/1": {"ip": "10.0.13.1/30", "area": 0},
                       "Lo0":   {"ip": "1.1.1.1/32"}},
        "ospf_process": 1,
        "ospf_area": 0,
        "neighbors": ["R2", "R3"],
        "running_config": "router ospf 1
 area 0 authentication
\!"
    },
    "R2": {
        "platform": "Cisco IOS",
        "interfaces": {"Gi0/0": {"ip": "10.0.12.2/30", "area": 0},
                       "Gi0/2": {"ip": "10.0.24.1/30", "area": 1},
                       "Lo0":   {"ip": "2.2.2.2/32"}},
        "ospf_process": 1,
        "ospf_area": 0,
        "neighbors": ["R1", "R4"],
        "running_config": "router ospf 1
\!"    # No auth yet
    },
    "R3": {
        "platform": "Cisco IOS-XE",
        "interfaces": {"Gi0/1": {"ip": "10.0.13.2/30", "area": 0},
                       "Lo0":   {"ip": "3.3.3.3/32"}},
        "ospf_process": 1,
        "ospf_area": 0,
        "neighbors": ["R1"],
        "running_config": "router ospf 1
 area 0 authentication
\!"
    },
}

print("Network inventory loaded:")
for router, info in NETWORK_INVENTORY.items():
    print(f"  {router}: {info["platform"]} | neighbors: {info["neighbors"]}")


---
## Demo 1 — Intent to Config: Close the Semantic Gap

The **semantic gap** is the distance between:
- What the engineer says: *"Enable OSPF MD5 authentication on all core routers"*
- What the network needs: 40 lines of platform-specific CLI commands that must match **exactly** on all neighbors simultaneously

This is where LLMs shine. They understand the business intent *and* know the
platform syntax differences between IOS, IOS-XE, and NX-OS.

```
Engineer intent: "Enable OSPF MD5 auth in area 0"
        │
   [LLM Translation Layer]
        │
  ┌─────┼─────┐
  R1    R2    R3
 IOS-XE IOS  IOS-XE
  │     │     │
 (different syntax per platform)
        │
  + Risk annotation: "Must change ALL neighbors simultaneously"
  + Rollback commands included
```


In [ ]:
# ── Intent to Configuration ───────────────────────────────────────────────────

def intent_to_config(intent: str, inventory: dict) -> dict:
    """
    Takes natural language intent and network inventory,
    returns annotated, platform-specific configuration per device.
    """
    # Build context from inventory
    inventory_summary = json.dumps(
        {r: {"platform": d["platform"],
             "ospf_process": d.get("ospf_process"),
             "current_auth": "authentication" in d.get("running_config", ""),
             "neighbors": d["neighbors"]}
         for r, d in inventory.items()},
        indent=2
    )

    prompt = f"""Network change intent: {intent}

Current network state:
{inventory_summary}

Generate configuration for EACH router. For each router:
1. Detect the platform (IOS vs IOS-XE — they have slightly different syntax)
2. Generate complete, correct CLI configuration
3. Note any risk or coordination requirement
4. Include rollback commands (how to undo this change)

Format as JSON:
{{
  "ROUTER_NAME": {{
    "platform": "...",
    "commands": ["line1", "line2", ...],
    "rollback_commands": ["undo_line1", ...],
    "risk_notes": "..."
  }}
}}
Return ONLY the JSON object.
"""
    raw = ask(prompt, model=SONNET)

    # Extract JSON
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {"error": "No JSON found", "raw": raw[:300]}
    return json.loads(match.group())


def display_config_plan(intent: str, inventory: dict):
    print(f"Intent: {intent}")
    print("=" * 60)

    configs = intent_to_config(intent, inventory)

    if "error" in configs:
        print(f"Error: {configs}")
        return configs

    for router, details in configs.items():
        print(f"\n[{router}] — {details.get('platform', 'Unknown')}")
        print("  Configuration commands:")
        for cmd in details.get("commands", []):
            print(f"    {cmd}")
        print("  Rollback commands:")
        for cmd in details.get("rollback_commands", []):
            print(f"    {cmd}")
        if details.get("risk_notes"):
            print(f"  ⚠ Risk: {details['risk_notes']}")

    return configs


configs = display_config_plan(
    "Enable OSPF MD5 authentication in area 0 on all core routers. Use key-id 1 and key 'Netw0rk$ecure'",
    NETWORK_INVENTORY
)


---
## Demo 2 — Risk Assessment: Score Every Change Before It Runs

Not all changes are equal. Adding a description to an interface is LOW risk.
Changing OSPF authentication on the backbone is HIGH risk.

The **Critic-Reviewer pattern** uses a second LLM call — separate from generation —
to critically evaluate the generated configuration. Think of it as the
**Change Advisory Board (CAB) review**, but automated.

Risk scoring criteria:
- **Scope**: How many devices does this change?
- **Reversibility**: Can we roll back in under 5 minutes?
- **Blast radius**: Which services depend on the changed component?
- **Coordination required**: Must all devices change simultaneously?
- **Timing sensitivity**: Does this need a maintenance window?

```
[Producer Agent] → generates config
        ↓
[Critic Agent] → scores risk, flags issues
        ↓
   RISK: LOW → auto-execute
   RISK: MEDIUM → log and execute with monitoring
   RISK: HIGH → block until human approves
```


In [ ]:
# ── Risk Assessment (Critic-Reviewer Pattern) ────────────────────────────────

@dataclass
class RiskReport:
    score: str          # LOW / MEDIUM / HIGH / CRITICAL
    score_int: int      # 1-4
    issues: list[str]
    coordination: bool  # True if all devices must change simultaneously
    needs_window: bool  # True if maintenance window required
    recommendation: str

def assess_risk(intent: str, configs: dict) -> RiskReport:
    """
    Critic agent independently evaluates the generated configurations.
    Completely separate from the generator — no shared context.
    """
    config_text = json.dumps(configs, indent=2)

    prompt = f"""You are a senior network change reviewer (the CAB/critic).

Change intent: {intent}

Generated configuration plan:
{config_text}

Critically evaluate this change for production safety. Check:
1. Does this affect multiple devices that must change simultaneously? (coordination risk)
2. Could a misconfiguration cause an outage? If yes, how many users/services?
3. Is the rollback fast (< 5 min) and clear?
4. Are there any IOS vs IOS-XE syntax differences that could cause failures?
5. Does this require a maintenance window?

Return ONLY this JSON:
{{
  "risk_score": "LOW|MEDIUM|HIGH|CRITICAL",
  "risk_int": 1,
  "issues": ["issue1", "issue2"],
  "coordination_required": true,
  "maintenance_window_required": false,
  "recommendation": "one sentence on what to do"
}}
"""
    raw = ask(prompt, model=SONNET,
              system="You are a strict network change reviewer. Be critical. Find issues.")

    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return RiskReport("HIGH", 3, ["Could not parse risk assessment"], True, True,
                         "Manual review required")
    data = json.loads(match.group())
    return RiskReport(
        score       = data.get("risk_score", "HIGH"),
        score_int   = data.get("risk_int", 3),
        issues      = data.get("issues", []),
        coordination= data.get("coordination_required", False),
        needs_window= data.get("maintenance_window_required", False),
        recommendation = data.get("recommendation", ""),
    )

def display_risk_report(intent: str, configs: dict):
    print(f"Assessing risk for: {intent}")
    print("=" * 60)
    report = assess_risk(intent, configs)

    colors = {"LOW": "✅", "MEDIUM": "⚠", "HIGH": "🔴", "CRITICAL": "🚨"}
    print(f"\nRisk Score: {colors.get(report.score, '?')} {report.score}")
    print(f"Coordination required: {'YES — simultaneous change on all devices' if report.coordination else 'No'}")
    print(f"Maintenance window: {'Required' if report.needs_window else 'Not required'}")
    print(f"\nIssues found ({len(report.issues)}):")
    for issue in report.issues:
        print(f"  • {issue}")
    print(f"\nRecommendation: {report.recommendation}")
    return report

risk = display_risk_report(
    "Enable OSPF MD5 authentication in area 0 on all core routers",
    configs
)


---
## Demo 3 — Digital Twin: Simulate Before You Touch Production

A **digital twin** is a software model of your network that mirrors the real state.
Before applying any change to production, you apply it to the twin first.

Think of it like BGP route simulation: you run the policy change in a lab,
verify the route table looks right, *then* deploy to production.

Our digital twin is a Python dictionary that represents:
- Device configurations (interfaces, OSPF, ACLs)
- Adjacency relationships (who neighbors whom)
- Service dependencies (what depends on which link)

When a change is applied to the twin, we detect:
- **Configuration conflicts** (IP overlap, missing commands on one side)
- **Adjacency impacts** (which OSPF neighbors will drop if auth changes)
- **Service impacts** (which services might be affected)


In [ ]:
# ── Digital Twin Simulation ───────────────────────────────────────────────────

class DigitalTwin:
    """
    A lightweight graph-based network model.
    Apply changes here first — if simulation passes, proceed to production.
    """

    def __init__(self, inventory: dict):
        # Deep copy so we can modify without affecting original
        self.state     = copy.deepcopy(inventory)
        self.changelog = []
        self.snapshots = []   # for rollback

    def snapshot(self):
        """Save current state — allows rollback."""
        self.snapshots.append(copy.deepcopy(self.state))
        return len(self.snapshots) - 1   # snapshot ID

    def rollback(self, snapshot_id: int = -1):
        """Revert to a previous snapshot."""
        if not self.snapshots:
            return False
        self.state = copy.deepcopy(self.snapshots[snapshot_id])
        self.changelog.append({"action": "ROLLBACK", "to_snapshot": snapshot_id})
        return True

    def apply_ospf_auth(self, routers: list[str], key_id: int, key: str) -> dict:
        """
        Simulates enabling OSPF MD5 authentication on given routers.
        Returns simulation result: adjacency impacts, conflicts, go/no-go.
        """
        snap_id = self.snapshot()
        impacts = []
        conflicts = []

        # Check which routers are neighbors but NOT in the change scope
        affected_pairs = set()
        for router in routers:
            device = self.state.get(router, {})
            for neighbor in device.get("neighbors", []):
                pair = tuple(sorted([router, neighbor]))
                affected_pairs.add(pair)
                if neighbor not in routers:
                    conflicts.append(
                        f"CONFLICT: {router} ↔ {neighbor} adjacency will DROP — "
                        f"{neighbor} is not in the change scope and has no auth"
                    )

        # Simulate applying auth to the chosen routers
        for router in routers:
            if router in self.state:
                self.state[router]["ospf_auth"] = {
                    "type": "MD5", "key_id": key_id, "key": key
                }
                self.changelog.append({
                    "action": "OSPF_AUTH_ENABLE",
                    "router": router, "key_id": key_id
                })

        # Adjacencies between changed routers are fine (both sides have auth)
        for r1, r2 in affected_pairs:
            if r1 in routers and r2 in routers:
                impacts.append(f"✓ {r1} ↔ {r2}: Both sides have auth — adjacency stays UP")
            else:
                impacts.append(f"✗ {r1} ↔ {r2}: Auth mismatch — adjacency will DROP")

        go = len(conflicts) == 0
        return {
            "go": go,
            "conflicts": conflicts,
            "adjacency_impacts": impacts,
            "snapshot_id": snap_id,
            "verdict": "SAFE TO DEPLOY" if go else "DO NOT DEPLOY — conflicts found",
        }

    def simulate_and_display(self, change_desc: str, routers: list[str],
                              key_id: int, key: str):
        print(f"Simulating on Digital Twin: {change_desc}")
        print(f"Scope: {routers}")
        print("=" * 60)

        result = self.apply_ospf_auth(routers, key_id, key)

        print(f"\nAdjacency impacts:")
        for impact in result["adjacency_impacts"]:
            print(f"  {impact}")

        if result["conflicts"]:
            print(f"\n⛔ Conflicts detected:")
            for c in result["conflicts"]:
                print(f"  {c}")
        else:
            print("\n✓ No conflicts found")

        print(f"\nVerdict: {result['verdict']}")
        return result


twin = DigitalTwin(NETWORK_INVENTORY)

# Scenario A: Partial rollout (DANGEROUS — will drop adjacencies)
print("SCENARIO A — Partial rollout (only R1 and R3, missing R2):")
result_a = twin.simulate_and_display(
    "Enable OSPF MD5 auth",
    routers=["R1", "R3"],   # R2 is missing!
    key_id=1, key="Netw0rk$ecure"
)

# Rollback twin to clean state
twin.rollback()
print("\n" + "─" * 60)

# Scenario B: Full rollout (SAFE — all routers included)
print("\nSCENARIO B — Full rollout (all routers R1, R2, R3):")
result_b = twin.simulate_and_display(
    "Enable OSPF MD5 auth",
    routers=["R1", "R2", "R3"],   # All neighbors included
    key_id=1, key="Netw0rk$ecure"
)


---
## Demo 4 — Human-in-the-Loop Approval Gate

For HIGH-risk changes, the agent **pauses** and waits for human approval before
any command touches a production device.

This is the automated equivalent of the **Change Advisory Board (CAB)**:
the AI does all the analysis, but a human makes the final go/no-go decision.

```
[Change Pipeline]
        │
   Risk: LOW  ──────────────────────→ Auto-execute
   Risk: MEDIUM ──────→ Log + Monitor → Execute
   Risk: HIGH  ──→ Generate Report → 🛑 WAIT for human → Execute or Abort
```

In real production, "waiting for human" means:
- Sending a Slack message with the change plan and risk report
- Waiting for a thumbs-up reaction or `/approve change-123`
- Or a webhook from your ticketing system (ServiceNow, Jira)

Here we simulate the approval interaction in the terminal.


In [ ]:
# ── Human-in-the-Loop Approval Gate ──────────────────────────────────────────

@dataclass
class ChangeRequest:
    id: str
    intent: str
    configs: dict
    risk: RiskReport
    twin_result: dict
    status: str = "PENDING"   # PENDING → APPROVED / REJECTED / AUTO_APPROVED

def format_change_summary(cr: ChangeRequest) -> str:
    """Format a human-readable change summary for approval."""
    lines = [
        f"CHANGE REQUEST — {cr.id}",
        "=" * 50,
        f"Intent    : {cr.intent}",
        f"Risk      : {cr.risk.score}",
        f"Simulation: {'PASSED' if cr.twin_result.get('go') else 'FAILED'}",
        f"Devices   : {list(cr.configs.keys()) if isinstance(cr.configs, dict) else 'N/A'}",
        "",
        "Issues found:",
    ]
    for issue in cr.risk.issues:
        lines.append(f"  • {issue}")
    lines.append(f"\nRecommendation: {cr.risk.recommendation}")
    return "\n".join(lines)


def approval_gate(cr: ChangeRequest, auto_approve_threshold: str = "MEDIUM") -> bool:
    """
    Routes the change based on risk:
    - LOW/MEDIUM (below threshold): auto-approve and log
    - HIGH/CRITICAL: block and request human decision

    Returns True if approved, False if rejected.
    """
    RISK_ORDER = {"LOW": 1, "MEDIUM": 2, "HIGH": 3, "CRITICAL": 4}
    threshold_int = RISK_ORDER.get(auto_approve_threshold, 2)

    print(format_change_summary(cr))
    print()

    # Auto-approve low-risk changes
    if cr.risk.score_int <= threshold_int:
        cr.status = "AUTO_APPROVED"
        print(f"✅ AUTO-APPROVED (risk={cr.risk.score} ≤ threshold={auto_approve_threshold})")
        print("   Change will execute without human intervention.")
        return True

    # Block high-risk changes — require human decision
    print(f"🔴 APPROVAL REQUIRED — Risk level is {cr.risk.score}")
    print()

    # In a real system this would be a Slack/webhook notification
    # For this demo we use input() — comment it out for non-interactive runs
    print("Options:")
    print("  [A] Approve — proceed with change as planned")
    print("  [R] Reject  — abort, do not touch production")
    print("  [M] Modify  — abort, go back and revise the plan")
    print()

    # ── Simulate human decision (change to input() for interactive use) ───────
    simulated_decision = "A"   # ← In real use: decision = input("Decision [A/R/M]: ").strip().upper()
    print(f"(Simulated human decision: {simulated_decision})")

    if simulated_decision == "A":
        cr.status = "APPROVED"
        print("\n✅ APPROVED by human operator. Proceeding with change.")
        return True
    elif simulated_decision == "R":
        cr.status = "REJECTED"
        print("\n⛔ REJECTED. Change aborted — no production devices touched.")
        return False
    else:
        cr.status = "REJECTED"
        print("\n🔄 MODIFY requested. Change sent back for revision.")
        return False


# Build a change request from our previous analysis
cr = ChangeRequest(
    id="CHG-2025-042",
    intent="Enable OSPF MD5 authentication in area 0 on all core routers",
    configs=configs,
    risk=risk,
    twin_result=result_b,
)

approved = approval_gate(cr, auto_approve_threshold="MEDIUM")
print(f"\nFinal status: {cr.status}")


---
## Demo 5 — Full Change Pipeline with Auto-Rollback

The complete automated change management system chains all previous demos:

```
Engineer types:  "Enable OSPF MD5 auth on core routers"
                         │
         1. Intent → Config (LLM generates per-device CLI)
                         │
         2. Risk Assessment (Critic agent scores: HIGH/MED/LOW)
                         │
         3. Digital Twin Simulation (conflict detection before production)
                         │
         4. Approval Gate (auto if LOW/MED, human gate if HIGH)
                         │
         5. Simulated Execution (with post-change verification)
                         │
         6. Auto-Rollback if verification fails
                         │
         7. Auto-generated Change Log (audit trail)
```

This replaces all five manual failure modes:
✓ Pre-validation, ✓ Impact analysis, ✓ Automated rollback,
✓ Verification, ✓ Auto-documentation


In [ ]:
# ── Full Change Automation Pipeline ──────────────────────────────────────────

def verify_change(intent: str, routers: list[str]) -> dict:
    """
    Post-change verification sweep.
    In production: runs 'show ip ospf neighbor' on all devices and checks adjacencies.
    Here: simulated with LLM.
    """
    prompt = f"""Post-change verification for: {intent}
Devices changed: {routers}

Simulate a verification sweep. Run these checks:
1. OSPF neighbor state on each device
2. Authentication type confirmed as MD5
3. All expected adjacencies in FULL state

Return JSON: {{"success": true/false, "results": {{"ROUTER": "status"}}, "issues": []}}
"""
    raw = ask(prompt, model=HAIKU)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    # Fallback
    return {"success": True, "results": {r: "FULL" for r in routers}, "issues": []}


def generate_change_log(intent: str, cr: ChangeRequest,
                        verification: dict, elapsed: float) -> str:
    """Auto-generate a structured change log — no manual documentation needed."""
    prompt = f"""Generate a concise change log entry for:

Intent: {intent}
Change ID: {cr.id}
Risk: {cr.risk.score}
Status: {cr.status}
Verification: {"PASSED" if verification.get('success') else "FAILED"}
Duration: {elapsed:.1f}s

Format:
CHANGE LOG — {cr.id}
Date/Time : (use today)
Engineer  : Automated AI Pipeline
Intent    : ...
Risk      : ...
Changes Applied: (bullet list per device)
Verification: ...
Status    : SUCCESS / FAILED
Lessons   : (one sentence)
"""
    return ask(prompt, model=HAIKU)


def run_full_pipeline(intent: str):
    print("=" * 70)
    print(f"NETWORK CHANGE PIPELINE")
    print(f"Intent: {intent}")
    print("=" * 70)
    start = time.time()
    pipeline_twin = DigitalTwin(NETWORK_INVENTORY)

    # ── Stage 1: Intent → Config ──────────────────────────────────────────────
    print("\n[1/6] Generating configuration from intent...")
    cfg = intent_to_config(intent, NETWORK_INVENTORY)
    if "error" in cfg:
        print(f"  ✗ Config generation failed: {cfg}")
        return
    print(f"  ✓ Generated config for: {list(cfg.keys())}")

    # ── Stage 2: Risk Assessment ──────────────────────────────────────────────
    print("\n[2/6] Running risk assessment (Critic agent)...")
    rpt = assess_risk(intent, cfg)
    print(f"  ✓ Risk score: {rpt.score}  |  Issues: {len(rpt.issues)}")

    # ── Stage 3: Digital Twin Simulation ─────────────────────────────────────
    print("\n[3/6] Simulating on Digital Twin...")
    routers = list(NETWORK_INVENTORY.keys())
    twin_result = pipeline_twin.apply_ospf_auth(routers, key_id=1, key="Netw0rk$ecure")
    verdict = "PASS" if twin_result["go"] else "FAIL"
    print(f"  ✓ Simulation: {verdict}")
    for impact in twin_result["adjacency_impacts"]:
        print(f"    {impact}")

    if not twin_result["go"]:
        print("  ✗ Simulation failed — conflicts detected. Pipeline ABORTED.")
        return

    # ── Stage 4: Approval Gate ────────────────────────────────────────────────
    print("\n[4/6] Approval gate...")
    change_req = ChangeRequest(
        id=f"CHG-AUTO-{int(time.time())}",
        intent=intent, configs=cfg, risk=rpt, twin_result=twin_result,
    )
    approved = approval_gate(change_req, auto_approve_threshold="MEDIUM")

    if not approved:
        print("\n  ✗ Change not approved. Pipeline ABORTED. No production devices touched.")
        return

    # ── Stage 5: Execute (simulated) ─────────────────────────────────────────
    print("\n[5/6] Executing change (simulated)...")
    for router in routers:
        cmds = cfg.get(router, {}).get("commands", [])
        print(f"  ▶ {router}: applying {len(cmds)} commands…")
        time.sleep(0.1)
        print(f"  ✓ {router}: done")

    # ── Stage 6: Verification + Auto-Rollback ─────────────────────────────────
    print("\n[6/6] Post-change verification sweep...")
    verification = verify_change(intent, routers)

    if verification.get("success"):
        print("  ✓ All routers verified — change successful")
        for router, status in verification.get("results", {}).items():
            print(f"    {router}: {status}")
    else:
        print("  ✗ Verification FAILED — triggering auto-rollback...")
        pipeline_twin.rollback()
        print("  ✓ Twin rolled back to pre-change state")
        print("  ⚠ In production: rollback commands would be pushed to all devices")
        for issue in verification.get("issues", []):
            print(f"    Issue: {issue}")

    # ── Auto-generated change log ─────────────────────────────────────────────
    elapsed = time.time() - start
    log = generate_change_log(intent, change_req, verification, elapsed)
    print("\n" + "=" * 70)
    print("AUTO-GENERATED CHANGE LOG")
    print("=" * 70)
    print(log)
    print("=" * 70)
    print(f"Pipeline complete in {elapsed:.1f}s")


run_full_pipeline(
    "Enable OSPF MD5 authentication in area 0 on all core routers. Key-id 1, key Netw0rk$ecure."
)


---
## Summary — The Five Failure Modes, Solved

| Failure Mode | Manual Process | AI Pipeline |
|---|---|---|
| **No pre-validation** | Mental review | LLM generates + Critic validates |
| **No impact analysis** | "Just a static route" | Digital Twin simulates blast radius |
| **No automated rollback** | Panic manual revert | Checkpoint + auto-rollback on failed verify |
| **No real-time verification** | One or two checks | Full post-change sweep, all devices |
| **Incomplete documentation** | Filled from memory | Auto-generated change log |

### Architecture Pattern

```
Intent (NL)
    │
[Translator — Sonnet]  →  Per-device, per-platform CLI config
    │
[Critic — Sonnet]      →  Risk score: LOW / MEDIUM / HIGH
    │
[Digital Twin]         →  Conflict detection without touching production
    │
[Approval Gate]        →  Auto-approve LOW, human gate for HIGH
    │
[Executor — Haiku]     →  Apply commands device by device
    │
[Verifier — Haiku]     →  Post-change sweep
    │
[Auto-Rollback]        →  Revert if verification fails
    │
[Logger — Haiku]       →  Audit trail, no manual docs
```

### Key Design Principles

1. **Separate generation from validation.** The Critic agent reviews blindly — it doesn't know it's reviewing its own sibling's output.
2. **Never skip the Digital Twin.** If simulation shows conflicts, abort before any production device is touched.
3. **Rollback must be in the plan.** The LLM generates rollback commands at the same time as the change commands — not as an afterthought.
4. **Automate documentation.** Engineers don't fill in change logs accurately under pressure — the pipeline does it automatically.

### What's Next

- **Chapter 22**: Observability — how to monitor AI agent behavior in production, detect when agents make wrong decisions, and build dashboards for your AI infrastructure

> The semantic gap between "I want OSPF authentication" and the 40 IOS-XE commands that implement it safely is exactly where human error lives. Closing this gap is what AI-assisted change automation does.
